# Phase 9: Standard-MLP Swin-MoE Kaggle Experiment

This notebook trains the Phase 9 `swin_standard_moe` model and evaluates it zero-shot.

In [ ]:
# ==========================================
# STEP 1: ENVIRONMENT SETUP
# ==========================================
import os
import sys
import shutil

# Delete old repo to ensure fresh code
!rm -rf vision_transformer_research
!git clone https://github.com/toqeer-ahmed/vision_tranformer_moe.git vision_transformer_research
os.chdir('vision_transformer_research')
sys.path.append(os.path.abspath('.'))

!pip install -r requirements.txt -q

In [ ]:
# ==========================================
# STEP 2: RESTRUCTURE DATASETS (BULLETPROOF)
# ==========================================
print("Setting up BUSI dataset paths...")
os.makedirs("data/medical_dataset/images", exist_ok=True)
os.makedirs("data/medical_dataset/masks", exist_ok=True)

# Search all of Kaggle's busi-dataset folder
for root_dir, _, files in os.walk("/kaggle/input/busi-dataset"):
    for file in files:
        if file.lower().endswith(('.png', '.jpg', '.jpeg')):
            src = os.path.join(root_dir, file)
            if "_mask" in file.lower():
                shutil.copy(src, os.path.join("data/medical_dataset/masks", file))
            else:
                shutil.copy(src, os.path.join("data/medical_dataset/images", file))

print(f"Total BUSI images: {len(os.listdir('data/medical_dataset/images'))}")
print(f"Total BUSI masks: {len(os.listdir('data/medical_dataset/masks'))}")

print("\nSetting up BUSBRA dataset paths for Zero-Shot...")
os.makedirs("data/busbra_dataset/images", exist_ok=True)
os.makedirs("data/busbra_dataset/masks", exist_ok=True)

# BUSBRA masks usually have the exact same name as the images but in a masks folder.
# Since BUSBRA on Kaggle might be separated or mixed, let's structure it safely.
busbra_root = "/kaggle/input/busbra-dataset"
for root_dir, _, files in os.walk(busbra_root):
    for file in files:
        if file.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp')):
            src = os.path.join(root_dir, file)
            # If the path contains 'mask' or the filename has 'mask', it's a mask.
            if "mask" in src.lower():
                shutil.copy(src, os.path.join("data/busbra_dataset/masks", file))
            else:
                shutil.copy(src, os.path.join("data/busbra_dataset/images", file))

print(f"Total BUSBRA images: {len(os.listdir('data/busbra_dataset/images'))}")
print(f"Total BUSBRA masks: {len(os.listdir('data/busbra_dataset/masks'))}")

In [ ]:
# ==========================================
# STEP 3: CONFIGURE EXPERIMENT
# ==========================================
config_yaml = """\
model:
  name: "swin_standard_moe"
  backbone: "microsoft/swin-tiny-patch4-window7-224"
  num_classes: 3

moe:
  num_experts: 4
  top_k: 2
  noisy_gating: true
  balance_loss_coef: 0.01
  warm_init: true

training:
  batch_size: 32
  learning_rate: 0.0001
  weight_decay: 0.01
  num_epochs: 50
  early_stopping_patience: 10
  img_size: 224
  seed: 42
"""
os.makedirs("configs", exist_ok=True)
with open("configs/swin_standard_moe_kaggle.yaml", "w") as f:
    f.write(config_yaml)
print("Configuration saved.")

In [ ]:
# ==========================================
# STEP 4: FULL TRAINING
# ==========================================
print("Starting Full Training...")
!python training/train_swin_standard_moe.py --config configs/swin_standard_moe_kaggle.yaml --warm-init

In [ ]:
# ==========================================
# STEP 5: ZERO-SHOT EVALUATION ON BUS-BRA
# ==========================================
import torch
import yaml
import glob
import json
from models.swin import SwinSegmentation, replace_swin_ffn_with_standard_moe
from datasets.medical_dataset import get_medical_dataloaders
from evaluation.metrics import SegmentationMetricAccumulator

# Find the best checkpoint from the training run
ckpt_paths = glob.glob("outputs/swin_standard_moe_*/checkpoints/best_model.pth")
if not ckpt_paths:
    print("Error: Could not find trained checkpoint!")
else:
    best_ckpt = ckpt_paths[-1]
    print(f"Evaluating checkpoint: {best_ckpt}")
    
    # Load config and model
    with open("configs/swin_standard_moe_kaggle.yaml", "r") as f:
        config = yaml.safe_load(f)
    
    model = SwinSegmentation(num_classes=3, pretrained=False)
    replace_swin_ffn_with_standard_moe(model, config["moe"])
    
    checkpoint = torch.load(best_ckpt, map_location="cuda")
    if "state_dict" in checkpoint:
        model.load_state_dict(checkpoint["state_dict"], strict=False)
    elif "model_state_dict" in checkpoint:
        model.load_state_dict(checkpoint["model_state_dict"], strict=False)
    else:
        model.load_state_dict(checkpoint, strict=False)
    
    model = model.cuda()
    model.eval()
    
    # Load BUSBRA Zero-Shot dataset. BUSBRA doesn't have train/val split in our dataloader out of the box, 
    # so we just use the test_loader which returns the whole dataset if we configure it right.
    # We will use the raw dataset class instead of the split loader to evaluate the entire BUSBRA.
    from datasets.medical_dataset import MedicalImageMaskDataset, get_medical_transforms
    from torch.utils.data import DataLoader
    
    transform = get_medical_transforms(224, is_train=False)
    busbra_dataset = MedicalImageMaskDataset("data/busbra_dataset", transform=transform)
    busbra_loader = DataLoader(busbra_dataset, batch_size=16, shuffle=False, num_workers=2)
    
    accumulator = SegmentationMetricAccumulator(num_classes=3)
    
    print("Running Zero-Shot Inference on BUSBRA...")
    with torch.no_grad():
        for images, targets in busbra_loader:
            images, targets = images.cuda(), targets.cuda()
            with torch.amp.autocast("cuda"):
                logits = model(images)
            preds = logits.argmax(dim=1)
            accumulator.update(preds, targets)
            
    metrics = accumulator.compute()
    print("\n=== FINAL ZERO-SHOT METRICS ===")
    for k, v in metrics.items():
        print(f"{k}: {v:.4f}")
        
    os.makedirs("zero_shot_results", exist_ok=True)
    with open("zero_shot_results/zero_shot_metrics.json", "w") as f:
        json.dump(metrics, f, indent=4)
    print("\nResults saved to zero_shot_results/zero_shot_metrics.json")